# NutriVision CNN Food-101 101 Kelas + AKG + Nutrition Table Cleaned + Gemini API

Notebook ini adalah versi **CNN 101 kelas Food-101** dengan:
- Dataset gambar Food-101 full **101 kelas**
- Model utama **Custom CNN dari nol** menggunakan TensorFlow Functional API
- Nutrisi makanan dari file baru `nutrition_table_cleaned.csv`
- Acuan kebutuhan gizi harian dari `akg_normal.csv`, `akg_pregnant.csv`, dan `akg_breastfeeding.csv`
- Custom Layer, Custom Loss, dan Custom Callback
- Export `.keras` dan SavedModel


In [ ]:
!pip -q install tensorflow-datasets psutil google-genai

import os
import gc
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import psutil
import tensorflow as tf
import tensorflow_datasets as tfds

print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

try:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision aktif.")
except Exception as e:
    print("Mixed precision tidak aktif:", e)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


TensorFlow: 2.20.0
TFDS: 4.9.10
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision aktif.


In [ ]:
# ==================
# Konfigurasi utama
# ==================

CLASS_LIMIT = 101
NUM_CLASSES = 101

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 100

BASE_LR = 3e-4
WEIGHT_DECAY = 1e-4

TARGET_ACCURACY = 0.85
TARGET_MAE = 0.02

SHUFFLE_BUFFER = 2048
PARALLEL_CALLS = tf.data.AUTOTUNE
PREFETCH_BUFFER = tf.data.AUTOTUNE

PROJECT_DIR = Path("/content/nutrivision_cnn_food101_akg")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = str(PROJECT_DIR / "best_nutrivision_cnn_food101_akg.keras")
FINAL_MODEL_PATH = str(PROJECT_DIR / "final_nutrivision_cnn_food101_akg.keras")
SAVED_MODEL_DIR = str(PROJECT_DIR / "saved_model_nutrivision_cnn_food101_akg")

CLASS_NAMES_PATH = str(PROJECT_DIR / "class_names.json")
NUTRITION_TABLE_PATH = str(PROJECT_DIR / "nutrition_table_cleaned.csv")
AKG_NORMAL_PATH = str(PROJECT_DIR / "akg_normal.csv")
AKG_PREGNANT_PATH = str(PROJECT_DIR / "akg_pregnant.csv")
AKG_BREASTFEEDING_PATH = str(PROJECT_DIR / "akg_breastfeeding.csv")

def show_ram(prefix="RAM"):
    mem = psutil.virtual_memory()
    print(
        f"[{prefix}] used={mem.used/1024**3:.2f}/{mem.total/1024**3:.2f} GB, "
        f"available={mem.available/1024**3:.2f} GB, percent={mem.percent:.1f}%"
    )

show_ram("Awal")


[Awal] used=1.82/167.05 GB, available=163.89 GB, percent=1.9%


In [ ]:
# ===================
# Membuat file CSV
# ===================

# File AKG
AKG_NORMAL_CSV = 'age_category,age_group,body_weight,height,calories,protein,total_fat,omega_3,omega_6,carbohydrates,dietary_fiber,water,vitamin_d,vitamin_e,vitamin_k,thiamin,riboflavin,niacin,pantothenic_acid,vitamin_b6,folate_dfe,vitamin_b12,biotin,choline,vitamin_c,calcium,phosphorus,magnesium,iron,iodine,zinc,selenium,manganese,chromium,potassium,sodium,chlorine,copper,vitamin_a,min_age,max_age,preg_month_min,preg_month_max,bf_month_min,bf_month_max\ninfants_children,0-5 months,6.0,60.0,550.0,9.0,31.0,0.5,4.4,59.0,0.0,700.0,10.0,0.004,5.0,0.2,0.3,2.0,1.7,0.1,80.0,0.4,5.0,125.0,40.0,200.0,100.0,30.0,0.3,90.0,1.1,7.0,3.0,0.2,400.0,120.0,180.0,0.2,375.0,0.0,0.4166666666666667,0.0,0.0,0.0,0.0\ninfants_children,6-11 months,9.0,72.0,800.0,15.0,35.0,0.5,4.4,105.0,11.0,900.0,10.0,0.005,10.0,0.3,0.4,4.0,1.8,0.3,80.0,1.5,6.0,150.0,50.0,270.0,275.0,55.0,11.0,120.0,3.0,10.0,0.7,6.0,700.0,370.0,570.0,0.22,400.0,0.5,0.9166666666666666,0.0,0.0,0.0,0.0\ninfants_children,1-3 years,13.0,92.0,1350.0,20.0,45.0,0.7,7.0,215.0,19.0,1150.0,15.0,0.006,15.0,0.5,0.5,6.0,2.0,0.5,160.0,1.5,8.0,200.0,40.0,650.0,460.0,65.0,7.0,90.0,3.0,18.0,1.2,14.0,2600.0,800.0,1200.0,0.34,400.0,1.0,3.0,0.0,0.0,0.0,0.0\ninfants_children,4-6 years,19.0,113.0,1400.0,25.0,50.0,0.9,10.0,220.0,20.0,1450.0,15.0,0.007,20.0,0.6,0.6,8.0,3.0,0.6,200.0,1.5,12.0,250.0,45.0,1000.0,500.0,95.0,10.0,120.0,5.0,21.0,1.5,16.0,2700.0,900.0,1300.0,0.44,450.0,4.0,6.0,0.0,0.0,0.0,0.0\ninfants_children,7-9 years,27.0,130.0,1650.0,40.0,55.0,0.9,10.0,250.0,23.0,1650.0,15.0,0.008,25.0,0.9,0.9,10.0,4.0,1.0,300.0,2.0,12.0,375.0,45.0,1000.0,500.0,135.0,10.0,120.0,5.0,22.0,1.7,21.0,3200.0,1000.0,1500.0,0.57,500.0,7.0,9.0,0.0,0.0,0.0,0.0\nmale,10-12 years,36.0,145.0,2000.0,50.0,65.0,1.2,12.0,300.0,28.0,1850.0,15.0,0.011,35.0,1.1,1.3,12.0,5.0,1.3,400.0,3.5,20.0,375.0,50.0,1200.0,1250.0,160.0,8.0,120.0,8.0,22.0,1.9,28.0,3900.0,1300.0,1900.0,0.7,600.0,10.0,12.0,0.0,0.0,0.0,0.0\nmale,13-15 years,50.0,163.0,2400.0,70.0,80.0,1.6,16.0,350.0,34.0,2100.0,15.0,0.015,55.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,25.0,550.0,75.0,1200.0,1250.0,225.0,11.0,150.0,11.0,30.0,2.2,36.0,4800.0,1500.0,2300.0,0.795,600.0,13.0,15.0,0.0,0.0,0.0,0.0\nmale,16-18 years,60.0,168.0,2650.0,75.0,85.0,1.6,16.0,400.0,37.0,2300.0,15.0,0.015,55.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1200.0,1250.0,270.0,11.0,150.0,11.0,36.0,2.3,41.0,5300.0,1700.0,2500.0,0.89,700.0,16.0,18.0,0.0,0.0,0.0,0.0\nmale,19-29 years,60.0,168.0,2650.0,65.0,75.0,1.6,17.0,430.0,37.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1000.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,36.0,4700.0,1500.0,2250.0,0.9,650.0,19.0,29.0,0.0,0.0,0.0,0.0\nmale,30-49 years,60.0,166.0,2550.0,65.0,70.0,1.6,17.0,415.0,36.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1000.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,34.0,4700.0,1500.0,2250.0,0.9,650.0,30.0,49.0,0.0,0.0,0.0,0.0\nmale,50-64 years,60.0,166.0,2150.0,65.0,60.0,1.6,14.0,340.0,30.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,29.0,4700.0,1300.0,2100.0,0.9,650.0,50.0,64.0,0.0,0.0,0.0,0.0\nmale,65-80 years,58.0,164.0,1800.0,64.0,50.0,1.6,14.0,275.0,25.0,1800.0,20.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,350.0,9.0,150.0,11.0,29.0,2.3,24.0,4700.0,1100.0,1900.0,0.9,650.0,65.0,80.0,0.0,0.0,0.0,0.0\nmale,80+ years,58.0,164.0,1600.0,64.0,45.0,1.6,14.0,235.0,22.0,1600.0,20.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,350.0,9.0,150.0,11.0,29.0,2.3,21.0,4700.0,1000.0,1600.0,0.9,650.0,80.0,100.0,0.0,0.0,0.0,0.0\nfemale,10-12 years,38.0,147.0,1900.0,55.0,65.0,1.0,10.0,280.0,27.0,1850.0,15.0,0.015,35.0,1.0,1.0,12.0,5.0,1.2,400.0,3.5,20.0,375.0,50.0,1200.0,1250.0,170.0,8.0,120.0,8.0,19.0,1.6,26.0,4400.0,1400.0,2100.0,0.7,600.0,10.0,12.0,0.0,0.0,0.0,0.0\nfemale,13-15 years,48.0,156.0,2050.0,65.0,70.0,1.1,11.0,300.0,29.0,2100.0,15.0,0.015,55.0,1.1,1.0,14.0,5.0,1.2,400.0,4.0,25.0,400.0,65.0,1200.0,1250.0,220.0,15.0,150.0,9.0,24.0,1.6,27.0,4800.0,1500.0,2300.0,0.7,600.0,13.0,15.0,0.0,0.0,0.0,0.0\nfemale,16-18 years,52.0,159.0,2100.0,65.0,70.0,1.1,11.0,300.0,29.0,2150.0,15.0,0.015,55.0,1.1,1.0,14.0,5.0,1.2,400.0,4.0,30.0,425.0,75.0,1200.0,1250.0,230.0,15.0,150.0,9.0,26.0,1.8,29.0,5000.0,1600.0,2400.0,0.89,600.0,16.0,18.0,0.0,0.0,0.0,0.0\nfemale,19-29 years,55.0,159.0,2250.0,60.0,65.0,1.1,12.0,360.0,32.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.3,400.0,4.0,30.0,425.0,75.0,1000.0,700.0,330.0,18.0,150.0,9.0,26.0,1.8,29.0,5000.0,1600.0,2400.0,0.89,600.0,19.0,29.0,0.0,0.0,0.0,0.0\nfemale,30-49 years,56.0,158.0,2150.0,60.0,60.0,1.1,12.0,340.0,30.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.3,400.0,4.0,30.0,425.0,75.0,1000.0,700.0,340.0,18.0,150.0,8.0,24.0,1.8,30.0,4700.0,1500.0,2250.0,0.9,600.0,30.0,49.0,0.0,0.0,0.0,0.0\nfemale,50-64 years,56.0,158.0,1800.0,60.0,50.0,1.1,11.0,280.0,25.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,340.0,8.0,150.0,8.0,25.0,1.8,24.0,4700.0,1400.0,2100.0,0.9,600.0,50.0,64.0,0.0,0.0,0.0,0.0\nfemale,65-80 years,53.0,157.0,1550.0,58.0,45.0,1.1,11.0,230.0,22.0,1550.0,20.0,0.02,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,320.0,8.0,150.0,8.0,24.0,1.8,21.0,4700.0,1200.0,1900.0,0.9,600.0,65.0,80.0,0.0,0.0,0.0,0.0\nfemale,80+ years,53.0,157.0,1400.0,58.0,40.0,1.1,11.0,200.0,20.0,1400.0,20.0,0.02,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,320.0,8.0,150.0,8.0,24.0,1.8,19.0,4700.0,1000.0,1600.0,0.9,600.0,80.0,100.0,0.0,0.0,0.0,0.0\npregnant,trimester 1,,,180.0,1.0,2.3,0.3,2.0,25.0,3.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,0.0,70.0,2.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,0.0,3.0,0.0,0.0\npregnant,trimester 2,,,300.0,10.0,2.3,0.3,2.0,40.0,4.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,9.0,70.0,4.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,4.0,6.0,0.0,0.0\npregnant,trimester 3,,,300.0,30.0,2.3,0.3,2.0,40.0,4.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,9.0,70.0,4.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,7.0,9.0,0.0,0.0\nbreastfeeding,first 6 months,,,330.0,20.0,2.2,0.2,2.0,45.0,5.0,800.0,0.0,0.004,0.0,0.4,0.5,3.0,2.0,0.6,100.0,1.0,5.0,125.0,45.0,200.0,0.0,0.0,0.0,140.0,5.0,10.0,0.8,20.0,400.0,0.0,400.0,0.4,350.0,13.0,49.0,0.0,0.0,0.0,6.0\nbreastfeeding,second 6 months,,,400.0,15.0,2.2,0.2,2.0,55.0,6.0,650.0,0.0,0.004,0.0,0.4,0.5,3.0,2.0,0.6,100.0,1.0,5.0,125.0,45.0,200.0,0.0,0.0,0.0,140.0,5.0,10.0,0.8,20.0,400.0,0.0,400.0,0.4,350.0,13.0,49.0,0.0,0.0,7.0,12.0\n'
AKG_PREGNANT_CSV = 'age_category,age_group,body_weight,height,calories,protein,total_fat,omega_3,omega_6,carbohydrates,dietary_fiber,water,vitamin_d,vitamin_e,vitamin_k,thiamin,riboflavin,niacin,pantothenic_acid,vitamin_b6,folate_dfe,vitamin_b12,biotin,choline,vitamin_c,calcium,phosphorus,magnesium,iron,iodine,zinc,selenium,manganese,chromium,potassium,sodium,chlorine,copper,vitamin_a,min_age,max_age,preg_month_min,preg_month_max,bf_month_min,bf_month_max\ninfants_children,0-5 months,6.0,60.0,550.0,9.0,31.0,0.5,4.4,59.0,0.0,700.0,10.0,0.004,5.0,0.2,0.3,2.0,1.7,0.1,80.0,0.4,5.0,125.0,40.0,200.0,100.0,30.0,0.3,90.0,1.1,7.0,3.0,0.2,400.0,120.0,180.0,0.2,375.0,0.0,0.4166666666666667,0.0,0.0,0.0,0.0\ninfants_children,6-11 months,9.0,72.0,800.0,15.0,35.0,0.5,4.4,105.0,11.0,900.0,10.0,0.005,10.0,0.3,0.4,4.0,1.8,0.3,80.0,1.5,6.0,150.0,50.0,270.0,275.0,55.0,11.0,120.0,3.0,10.0,0.7,6.0,700.0,370.0,570.0,0.22,400.0,0.5,0.9166666666666666,0.0,0.0,0.0,0.0\ninfants_children,1-3 years,13.0,92.0,1350.0,20.0,45.0,0.7,7.0,215.0,19.0,1150.0,15.0,0.006,15.0,0.5,0.5,6.0,2.0,0.5,160.0,1.5,8.0,200.0,40.0,650.0,460.0,65.0,7.0,90.0,3.0,18.0,1.2,14.0,2600.0,800.0,1200.0,0.34,400.0,1.0,3.0,0.0,0.0,0.0,0.0\ninfants_children,4-6 years,19.0,113.0,1400.0,25.0,50.0,0.9,10.0,220.0,20.0,1450.0,15.0,0.007,20.0,0.6,0.6,8.0,3.0,0.6,200.0,1.5,12.0,250.0,45.0,1000.0,500.0,95.0,10.0,120.0,5.0,21.0,1.5,16.0,2700.0,900.0,1300.0,0.44,450.0,4.0,6.0,0.0,0.0,0.0,0.0\ninfants_children,7-9 years,27.0,130.0,1650.0,40.0,55.0,0.9,10.0,250.0,23.0,1650.0,15.0,0.008,25.0,0.9,0.9,10.0,4.0,1.0,300.0,2.0,12.0,375.0,45.0,1000.0,500.0,135.0,10.0,120.0,5.0,22.0,1.7,21.0,3200.0,1000.0,1500.0,0.57,500.0,7.0,9.0,0.0,0.0,0.0,0.0\nmale,10-12 years,36.0,145.0,2000.0,50.0,65.0,1.2,12.0,300.0,28.0,1850.0,15.0,0.011,35.0,1.1,1.3,12.0,5.0,1.3,400.0,3.5,20.0,375.0,50.0,1200.0,1250.0,160.0,8.0,120.0,8.0,22.0,1.9,28.0,3900.0,1300.0,1900.0,0.7,600.0,10.0,12.0,0.0,0.0,0.0,0.0\nmale,13-15 years,50.0,163.0,2400.0,70.0,80.0,1.6,16.0,350.0,34.0,2100.0,15.0,0.015,55.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,25.0,550.0,75.0,1200.0,1250.0,225.0,11.0,150.0,11.0,30.0,2.2,36.0,4800.0,1500.0,2300.0,0.795,600.0,13.0,15.0,0.0,0.0,0.0,0.0\nmale,16-18 years,60.0,168.0,2650.0,75.0,85.0,1.6,16.0,400.0,37.0,2300.0,15.0,0.015,55.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1200.0,1250.0,270.0,11.0,150.0,11.0,36.0,2.3,41.0,5300.0,1700.0,2500.0,0.89,700.0,16.0,18.0,0.0,0.0,0.0,0.0\nmale,19-29 years,60.0,168.0,2650.0,65.0,75.0,1.6,17.0,430.0,37.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1000.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,36.0,4700.0,1500.0,2250.0,0.9,650.0,19.0,29.0,0.0,0.0,0.0,0.0\nmale,30-49 years,60.0,166.0,2550.0,65.0,70.0,1.6,17.0,415.0,36.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1000.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,34.0,4700.0,1500.0,2250.0,0.9,650.0,30.0,49.0,0.0,0.0,0.0,0.0\nmale,50-64 years,60.0,166.0,2150.0,65.0,60.0,1.6,14.0,340.0,30.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,29.0,4700.0,1300.0,2100.0,0.9,650.0,50.0,64.0,0.0,0.0,0.0,0.0\nmale,65-80 years,58.0,164.0,1800.0,64.0,50.0,1.6,14.0,275.0,25.0,1800.0,20.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,350.0,9.0,150.0,11.0,29.0,2.3,24.0,4700.0,1100.0,1900.0,0.9,650.0,65.0,80.0,0.0,0.0,0.0,0.0\nmale,80+ years,58.0,164.0,1600.0,64.0,45.0,1.6,14.0,235.0,22.0,1600.0,20.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,350.0,9.0,150.0,11.0,29.0,2.3,21.0,4700.0,1000.0,1600.0,0.9,650.0,80.0,100.0,0.0,0.0,0.0,0.0\nfemale,10-12 years,38.0,147.0,1900.0,55.0,65.0,1.0,10.0,280.0,27.0,1850.0,15.0,0.015,35.0,1.0,1.0,12.0,5.0,1.2,400.0,3.5,20.0,375.0,50.0,1200.0,1250.0,170.0,8.0,120.0,8.0,19.0,1.6,26.0,4400.0,1400.0,2100.0,0.7,600.0,10.0,12.0,0.0,0.0,0.0,0.0\nfemale,13-15 years,48.0,156.0,2050.0,65.0,70.0,1.1,11.0,300.0,29.0,2100.0,15.0,0.015,55.0,1.1,1.0,14.0,5.0,1.2,400.0,4.0,25.0,400.0,65.0,1200.0,1250.0,220.0,15.0,150.0,9.0,24.0,1.6,27.0,4800.0,1500.0,2300.0,0.7,600.0,13.0,15.0,0.0,0.0,0.0,0.0\nfemale,16-18 years,52.0,159.0,2100.0,65.0,70.0,1.1,11.0,300.0,29.0,2150.0,15.0,0.015,55.0,1.1,1.0,14.0,5.0,1.2,400.0,4.0,30.0,425.0,75.0,1200.0,1250.0,230.0,15.0,150.0,9.0,26.0,1.8,29.0,5000.0,1600.0,2400.0,0.89,600.0,16.0,18.0,0.0,0.0,0.0,0.0\nfemale,19-29 years,55.0,159.0,2250.0,60.0,65.0,1.1,12.0,360.0,32.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.3,400.0,4.0,30.0,425.0,75.0,1000.0,700.0,330.0,18.0,150.0,9.0,26.0,1.8,29.0,5000.0,1600.0,2400.0,0.89,600.0,19.0,29.0,0.0,0.0,0.0,0.0\nfemale,30-49 years,56.0,158.0,2150.0,60.0,60.0,1.1,12.0,340.0,30.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.3,400.0,4.0,30.0,425.0,75.0,1000.0,700.0,340.0,18.0,150.0,8.0,24.0,1.8,30.0,4700.0,1500.0,2250.0,0.9,600.0,30.0,49.0,0.0,0.0,0.0,0.0\nfemale,50-64 years,56.0,158.0,1800.0,60.0,50.0,1.1,11.0,280.0,25.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,340.0,8.0,150.0,8.0,25.0,1.8,24.0,4700.0,1400.0,2100.0,0.9,600.0,50.0,64.0,0.0,0.0,0.0,0.0\nfemale,65-80 years,53.0,157.0,1550.0,58.0,45.0,1.1,11.0,230.0,22.0,1550.0,20.0,0.02,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,320.0,8.0,150.0,8.0,24.0,1.8,21.0,4700.0,1200.0,1900.0,0.9,600.0,65.0,80.0,0.0,0.0,0.0,0.0\nfemale,80+ years,53.0,157.0,1400.0,58.0,40.0,1.1,11.0,200.0,20.0,1400.0,20.0,0.02,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,320.0,8.0,150.0,8.0,24.0,1.8,19.0,4700.0,1000.0,1600.0,0.9,600.0,80.0,100.0,0.0,0.0,0.0,0.0\npregnant,trimester 1,,,180.0,1.0,2.3,0.3,2.0,25.0,3.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,0.0,70.0,2.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,0.0,3.0,0.0,0.0\npregnant,trimester 2,,,300.0,10.0,2.3,0.3,2.0,40.0,4.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,9.0,70.0,4.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,4.0,6.0,0.0,0.0\npregnant,trimester 3,,,300.0,30.0,2.3,0.3,2.0,40.0,4.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,9.0,70.0,4.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,7.0,9.0,0.0,0.0\nbreastfeeding,first 6 months,,,330.0,20.0,2.2,0.2,2.0,45.0,5.0,800.0,0.0,0.004,0.0,0.4,0.5,3.0,2.0,0.6,100.0,1.0,5.0,125.0,45.0,200.0,0.0,0.0,0.0,140.0,5.0,10.0,0.8,20.0,400.0,0.0,400.0,0.4,350.0,13.0,49.0,0.0,0.0,0.0,6.0\nbreastfeeding,second 6 months,,,400.0,15.0,2.2,0.2,2.0,55.0,6.0,650.0,0.0,0.004,0.0,0.4,0.5,3.0,2.0,0.6,100.0,1.0,5.0,125.0,45.0,200.0,0.0,0.0,0.0,140.0,5.0,10.0,0.8,20.0,400.0,0.0,400.0,0.4,350.0,13.0,49.0,0.0,0.0,7.0,12.0\n'
AKG_BREASTFEEDING_CSV = 'age_category,age_group,body_weight,height,calories,protein,total_fat,omega_3,omega_6,carbohydrates,dietary_fiber,water,vitamin_d,vitamin_e,vitamin_k,thiamin,riboflavin,niacin,pantothenic_acid,vitamin_b6,folate_dfe,vitamin_b12,biotin,choline,vitamin_c,calcium,phosphorus,magnesium,iron,iodine,zinc,selenium,manganese,chromium,potassium,sodium,chlorine,copper,vitamin_a,min_age,max_age,preg_month_min,preg_month_max,bf_month_min,bf_month_max\ninfants_children,0-5 months,6.0,60.0,550.0,9.0,31.0,0.5,4.4,59.0,0.0,700.0,10.0,0.004,5.0,0.2,0.3,2.0,1.7,0.1,80.0,0.4,5.0,125.0,40.0,200.0,100.0,30.0,0.3,90.0,1.1,7.0,3.0,0.2,400.0,120.0,180.0,0.2,375.0,0.0,0.4166666666666667,0.0,0.0,0.0,0.0\ninfants_children,6-11 months,9.0,72.0,800.0,15.0,35.0,0.5,4.4,105.0,11.0,900.0,10.0,0.005,10.0,0.3,0.4,4.0,1.8,0.3,80.0,1.5,6.0,150.0,50.0,270.0,275.0,55.0,11.0,120.0,3.0,10.0,0.7,6.0,700.0,370.0,570.0,0.22,400.0,0.5,0.9166666666666666,0.0,0.0,0.0,0.0\ninfants_children,1-3 years,13.0,92.0,1350.0,20.0,45.0,0.7,7.0,215.0,19.0,1150.0,15.0,0.006,15.0,0.5,0.5,6.0,2.0,0.5,160.0,1.5,8.0,200.0,40.0,650.0,460.0,65.0,7.0,90.0,3.0,18.0,1.2,14.0,2600.0,800.0,1200.0,0.34,400.0,1.0,3.0,0.0,0.0,0.0,0.0\ninfants_children,4-6 years,19.0,113.0,1400.0,25.0,50.0,0.9,10.0,220.0,20.0,1450.0,15.0,0.007,20.0,0.6,0.6,8.0,3.0,0.6,200.0,1.5,12.0,250.0,45.0,1000.0,500.0,95.0,10.0,120.0,5.0,21.0,1.5,16.0,2700.0,900.0,1300.0,0.44,450.0,4.0,6.0,0.0,0.0,0.0,0.0\ninfants_children,7-9 years,27.0,130.0,1650.0,40.0,55.0,0.9,10.0,250.0,23.0,1650.0,15.0,0.008,25.0,0.9,0.9,10.0,4.0,1.0,300.0,2.0,12.0,375.0,45.0,1000.0,500.0,135.0,10.0,120.0,5.0,22.0,1.7,21.0,3200.0,1000.0,1500.0,0.57,500.0,7.0,9.0,0.0,0.0,0.0,0.0\nmale,10-12 years,36.0,145.0,2000.0,50.0,65.0,1.2,12.0,300.0,28.0,1850.0,15.0,0.011,35.0,1.1,1.3,12.0,5.0,1.3,400.0,3.5,20.0,375.0,50.0,1200.0,1250.0,160.0,8.0,120.0,8.0,22.0,1.9,28.0,3900.0,1300.0,1900.0,0.7,600.0,10.0,12.0,0.0,0.0,0.0,0.0\nmale,13-15 years,50.0,163.0,2400.0,70.0,80.0,1.6,16.0,350.0,34.0,2100.0,15.0,0.015,55.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,25.0,550.0,75.0,1200.0,1250.0,225.0,11.0,150.0,11.0,30.0,2.2,36.0,4800.0,1500.0,2300.0,0.795,600.0,13.0,15.0,0.0,0.0,0.0,0.0\nmale,16-18 years,60.0,168.0,2650.0,75.0,85.0,1.6,16.0,400.0,37.0,2300.0,15.0,0.015,55.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1200.0,1250.0,270.0,11.0,150.0,11.0,36.0,2.3,41.0,5300.0,1700.0,2500.0,0.89,700.0,16.0,18.0,0.0,0.0,0.0,0.0\nmale,19-29 years,60.0,168.0,2650.0,65.0,75.0,1.6,17.0,430.0,37.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1000.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,36.0,4700.0,1500.0,2250.0,0.9,650.0,19.0,29.0,0.0,0.0,0.0,0.0\nmale,30-49 years,60.0,166.0,2550.0,65.0,70.0,1.6,17.0,415.0,36.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.3,400.0,4.0,30.0,550.0,90.0,1000.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,34.0,4700.0,1500.0,2250.0,0.9,650.0,30.0,49.0,0.0,0.0,0.0,0.0\nmale,50-64 years,60.0,166.0,2150.0,65.0,60.0,1.6,14.0,340.0,30.0,2500.0,15.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,360.0,9.0,150.0,11.0,30.0,2.3,29.0,4700.0,1300.0,2100.0,0.9,650.0,50.0,64.0,0.0,0.0,0.0,0.0\nmale,65-80 years,58.0,164.0,1800.0,64.0,50.0,1.6,14.0,275.0,25.0,1800.0,20.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,350.0,9.0,150.0,11.0,29.0,2.3,24.0,4700.0,1100.0,1900.0,0.9,650.0,65.0,80.0,0.0,0.0,0.0,0.0\nmale,80+ years,58.0,164.0,1600.0,64.0,45.0,1.6,14.0,235.0,22.0,1600.0,20.0,0.015,65.0,1.2,1.3,16.0,5.0,1.7,400.0,4.0,30.0,550.0,90.0,1200.0,700.0,350.0,9.0,150.0,11.0,29.0,2.3,21.0,4700.0,1000.0,1600.0,0.9,650.0,80.0,100.0,0.0,0.0,0.0,0.0\nfemale,10-12 years,38.0,147.0,1900.0,55.0,65.0,1.0,10.0,280.0,27.0,1850.0,15.0,0.015,35.0,1.0,1.0,12.0,5.0,1.2,400.0,3.5,20.0,375.0,50.0,1200.0,1250.0,170.0,8.0,120.0,8.0,19.0,1.6,26.0,4400.0,1400.0,2100.0,0.7,600.0,10.0,12.0,0.0,0.0,0.0,0.0\nfemale,13-15 years,48.0,156.0,2050.0,65.0,70.0,1.1,11.0,300.0,29.0,2100.0,15.0,0.015,55.0,1.1,1.0,14.0,5.0,1.2,400.0,4.0,25.0,400.0,65.0,1200.0,1250.0,220.0,15.0,150.0,9.0,24.0,1.6,27.0,4800.0,1500.0,2300.0,0.7,600.0,13.0,15.0,0.0,0.0,0.0,0.0\nfemale,16-18 years,52.0,159.0,2100.0,65.0,70.0,1.1,11.0,300.0,29.0,2150.0,15.0,0.015,55.0,1.1,1.0,14.0,5.0,1.2,400.0,4.0,30.0,425.0,75.0,1200.0,1250.0,230.0,15.0,150.0,9.0,26.0,1.8,29.0,5000.0,1600.0,2400.0,0.89,600.0,16.0,18.0,0.0,0.0,0.0,0.0\nfemale,19-29 years,55.0,159.0,2250.0,60.0,65.0,1.1,12.0,360.0,32.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.3,400.0,4.0,30.0,425.0,75.0,1000.0,700.0,330.0,18.0,150.0,9.0,26.0,1.8,29.0,5000.0,1600.0,2400.0,0.89,600.0,19.0,29.0,0.0,0.0,0.0,0.0\nfemale,30-49 years,56.0,158.0,2150.0,60.0,60.0,1.1,12.0,340.0,30.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.3,400.0,4.0,30.0,425.0,75.0,1000.0,700.0,340.0,18.0,150.0,8.0,24.0,1.8,30.0,4700.0,1500.0,2250.0,0.9,600.0,30.0,49.0,0.0,0.0,0.0,0.0\nfemale,50-64 years,56.0,158.0,1800.0,60.0,50.0,1.1,11.0,280.0,25.0,2350.0,15.0,0.015,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,340.0,8.0,150.0,8.0,25.0,1.8,24.0,4700.0,1400.0,2100.0,0.9,600.0,50.0,64.0,0.0,0.0,0.0,0.0\nfemale,65-80 years,53.0,157.0,1550.0,58.0,45.0,1.1,11.0,230.0,22.0,1550.0,20.0,0.02,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,320.0,8.0,150.0,8.0,24.0,1.8,21.0,4700.0,1200.0,1900.0,0.9,600.0,65.0,80.0,0.0,0.0,0.0,0.0\nfemale,80+ years,53.0,157.0,1400.0,58.0,40.0,1.1,11.0,200.0,20.0,1400.0,20.0,0.02,55.0,1.1,1.1,14.0,5.0,1.5,400.0,4.0,30.0,425.0,75.0,1200.0,700.0,320.0,8.0,150.0,8.0,24.0,1.8,19.0,4700.0,1000.0,1600.0,0.9,600.0,80.0,100.0,0.0,0.0,0.0,0.0\npregnant,trimester 1,,,180.0,1.0,2.3,0.3,2.0,25.0,3.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,0.0,70.0,2.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,0.0,3.0,0.0,0.0\npregnant,trimester 2,,,300.0,10.0,2.3,0.3,2.0,40.0,4.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,9.0,70.0,4.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,4.0,6.0,0.0,0.0\npregnant,trimester 3,,,300.0,30.0,2.3,0.3,2.0,40.0,4.0,300.0,0.0,0.0,0.0,0.3,0.3,4.0,1.0,0.6,200.0,0.5,0.0,25.0,10.0,200.0,0.0,0.0,9.0,70.0,4.0,5.0,0.2,5.0,0.0,0.0,0.0,0.1,300.0,13.0,49.0,7.0,9.0,0.0,0.0\nbreastfeeding,first 6 months,,,330.0,20.0,2.2,0.2,2.0,45.0,5.0,800.0,0.0,0.004,0.0,0.4,0.5,3.0,2.0,0.6,100.0,1.0,5.0,125.0,45.0,200.0,0.0,0.0,0.0,140.0,5.0,10.0,0.8,20.0,400.0,0.0,400.0,0.4,350.0,13.0,49.0,0.0,0.0,0.0,6.0\nbreastfeeding,second 6 months,,,400.0,15.0,2.2,0.2,2.0,55.0,6.0,650.0,0.0,0.004,0.0,0.4,0.5,3.0,2.0,0.6,100.0,1.0,5.0,125.0,45.0,200.0,0.0,0.0,0.0,140.0,5.0,10.0,0.8,20.0,400.0,0.0,400.0,0.4,350.0,13.0,49.0,0.0,0.0,7.0,12.0\n'

NUTRITION_TABLE_CLEANED_CSV = 'class_name,calories_kcal,protein_g,carbs_g,fat_g,fiber_g,calcium_mg,iron_mg,vitamin_c_mg\napple_pie,360.5313801954288,8.448368289511425,51.01361675267777,16.04208587868602,1.8564759064282579,80.56433500165811,1.737752244686075,3.4859359741008538\nbaby_back_ribs,300.05039597298986,11.653377370461147,28.921126874768152,18.032969314448412,1.9818910723353031,79.00936585634116,1.8583843802388507,4.800256297983251\nbaklava,359.2286322453035,8.448368289511425,28.921126874768152,15.922832604042739,1.8564759064282579,89.82614104111661,1.799674831602202,3.4859359741008538\nbeef_carpaccio,301.7341705113248,11.653377370461147,27.987209803160567,17.759368109909023,1.982884843868884,78.56190718662586,1.8583843802388507,4.939264783892008\nbeef_tartare,304.03231709027176,11.653377370461147,28.921126874768152,18.151465632167216,1.9940646907192339,80.68512275302866,1.8583843802388507,5.284608870499544\nbeet_salad,184.9355374767775,9.7358962953177,24.983216023812336,9.824250046817461,2.115581452707587,78.0967797821437,1.782989295518366,6.091555792958139\nbeignets,366.1166639945323,8.448368289511425,52.257298162237824,16.215492003020817,1.8564759064282579,91.17903790621573,1.737752244686075,4.903376313800051\nbibimbap,235.92552382253155,9.65024669486757,28.892088760714305,9.915104292208147,2.0635878365080274,80.57106758804024,1.7852361969653556,4.903376313800051\nbread_pudding,356.85483670152206,8.448368289511425,28.921126874768152,16.03668127784029,1.8564759064282579,88.47573575010713,1.737752244686075,3.4859359741008538\nbreakfast_burrito,247.07096091720211,9.989117681055413,28.921126874768152,10.321189180455386,1.9938057055424951,78.62829264088928,1.7832375309240833,4.790141199707989\nbruschetta,177.18777047200805,10.157544655678349,24.786397387734933,9.999776574093673,2.115581452707587,78.8649943659361,1.7968993574938497,6.091555792958139\ncaesar_salad,180.84276587045562,9.925484117957959,24.865089438756822,10.002572150577883,2.115581452707587,79.33122846442112,1.810577917707378,6.091555792958139\ncannoli,258.45280188728594,8.448368289511425,51.87503098903181,15.98330605305138,1.8564759064282579,90.5159831859348,1.737752244686075,3.4859359741008538\ncaprese_salad,183.7846950493214,9.989117681055413,25.907611559133038,10.321189180455386,2.115581452707587,102.11542574279352,1.791107317772872,6.091555792958139\ncarrot_cake,359.93165603926775,8.448368289511425,51.495593526762704,16.089958958331156,1.9938057055424951,89.82606583725708,1.737752244686075,3.4859359741008538\nceviche,244.97488021613063,10.11804006007986,27.451895331794937,10.321189180455386,1.9710533817084424,80.56433500165811,1.8029249279408717,4.926679031497068\ncheesecake,355.1651867737488,9.026212954957836,52.798263226201016,16.030891488167622,1.8564759064282579,102.11542574279352,1.8583843802388507,3.4859359741008538\ncheese_plate,237.30310583489617,9.802796797902452,28.162679126830376,10.408807788525484,2.033445634724066,102.11542574279352,1.795046676353082,5.420867261664277\nchicken_curry,298.6341978560581,11.653377370461147,28.26178118287869,18.065516478073995,2.0041760190750573,77.2826402536433,1.8583843802388507,4.386595617431482\nchicken_quesadilla,303.229788892887,11.653377370461147,28.132347222435428,17.780246526298843,1.9245217009635935,79.81340135103675,1.8583843802388507,5.084211149457707\nchicken_wings,304.16736256453413,11.653377370461147,28.06540809271159,17.940526819850156,1.9914031242695303,80.41426357583208,1.8583843802388507,4.978979886752441\nchocolate_cake,258.45280188728594,8.448368289511425,52.15266352434931,16.097651476379625,1.8564759064282579,93.4903983605533,1.799674831602202,3.4859359741008538\nchocolate_mousse,359.3163068719972,8.448368289511425,52.0512137792469,16.125543089419004,1.8564759064282579,89.51352287070178,1.737752244686075,4.903376313800051\nchurros,359.84974789630223,8.448368289511425,52.171380177621174,15.749340125736095,1.8564759064282579,87.83516643199103,1.737752244686075,3.4859359741008538\nclam_chowder,238.74988797051802,10.451498965105001,27.30092618093905,9.91538987658752,1.9879836347610937,77.94185431614699,1.7965854285113598,4.92841572497064\nclub_sandwich,257.38842198178463,9.989117681055413,38.7599578911053,9.949213988676478,2.0144301756413485,79.04958894446833,1.8190390311473128,5.014267763399407\ncrab_cakes,358.8699755909345,11.653377370461147,51.4303615974424,16.293653834005426,1.8564759064282579,92.09078714588654,1.799674831602202,3.4859359741008538\ncreme_brulee,353.86893685134953,8.448368289511425,51.89990965361505,16.118733346597672,1.8564759064282579,91.64492239092496,1.737752244686075,3.4859359741008538\ncroque_madame,242.05367156240254,9.937256459865052,28.093834158970527,10.334607966773913,2.004635815283844,78.95219194699031,1.817915923435945,4.903376313800051\ncup_cakes,258.45280188728594,8.448368289511425,51.6312276487243,15.663904146557446,1.8564759064282579,89.91194674104347,1.737752244686075,3.4859359741008538\ndeviled_eggs,236.79468799025474,9.611106580437598,27.92325619604988,9.76591972647954,2.0262411998443346,80.56433500165811,1.809006126164812,5.114448360886657\ndonuts,360.4230273433859,8.448368289511425,51.535535111907585,16.031496406615524,1.8564759064282579,87.30273603979856,1.737752244686075,3.4859359741008538\ndumplings,258.45280188728594,10.467305732809253,37.977296326530535,9.971704756847961,1.9938057055424951,80.99138580109229,1.7864417999363886,4.903376313800051\nedamame,181.7244761632109,10.139984653080333,25.305617996348566,10.321189180455386,2.115581452707587,78.15518784857856,1.8131473294065599,6.091555792958139\neggs_benedict,258.45280188728594,9.578152926395061,28.921126874768152,10.030128261319103,1.9938057055424951,79.42643724160091,1.7780734281947281,4.903376313800051\nescargots,258.45280188728594,10.193580600979583,28.51882815252422,10.06240972266547,2.029466244836347,79.04429783652267,1.7898480524705354,4.906439554563394\nfalafel,334.1392749110119,9.42124196843921,29.791454835730555,17.99703124071748,2.0021794584682184,80.08891816934154,1.7913028963179014,4.929210802603236\nfilet_mignon,244.13570043842347,10.459877690033723,28.105421815601307,9.907066899652516,2.073620804038366,80.56433500165811,1.8118099150184686,5.590673739130064\nfish_and_chips,236.9060411063433,11.653377370461147,28.29190091332804,10.371805273690875,1.9344910998327454,81.27028103556883,1.799674831602202,4.125971249239031\nfoie_gras,300.81841963022936,9.989117681055413,27.399857294926335,17.993380673101353,1.9938057055424951,80.79453983564403,1.8583843802388507,4.868344989613088\nfrench_fries,333.3536498979904,9.908332598831619,29.56085858597053,18.37429738793484,2.115581452707587,81.51005914811442,1.800496516352371,4.472526363700489\nfrench_onion_soup,240.5354817669336,9.673921105906205,27.66715831692316,9.912672317400474,1.9971820066788577,82.17765836652502,1.7749467482834524,4.843171131561722\nfrench_toast,354.4392760578577,8.448368289511425,51.9297712220153,16.100976405534915,1.8564759064282579,88.58700520033453,1.737752244686075,3.4859359741008538\nfried_calamari,332.870500661546,9.769859832447992,30.218681823729302,17.936925345131552,2.025163864249224,80.56433500165811,1.8118571202594906,4.844209874594191\nfried_rice,329.9630048544591,10.150692143904005,38.11572337473205,17.764761393174158,2.0372147048216562,80.06945158617577,1.8191867570094633,5.403193157165383\nfrozen_yogurt,240.16404282170726,9.612927989727442,27.808706802080195,25.285073314374774,2.002218656653438,80.63660178141068,1.8027531187266146,4.695751074912044\ngarlic_bread,256.4714880817426,9.989117681055413,38.932440154010195,9.609419786771054,2.009990917892456,78.758356625423,1.799674831602202,4.901018951035699\ngnocchi,261.1372763859446,9.989117681055413,37.886899051346774,10.110228251504312,1.9469872953694178,81.71804424799393,1.8001157826166023,4.903376313800051\ngreek_salad,181.98590484836237,9.781886925765068,24.814385695212422,9.96977604426478,2.115581452707587,79.2353915779495,1.799674831602202,6.091555792958139\ngrilled_cheese_sandwich,261.8093330526179,10.000820470500255,36.78926067163508,9.855447810598667,2.0283337696819377,102.11542574279352,1.8179132451202396,4.925582506092468\ngrilled_salmon,258.45280188728594,11.653377370461147,27.98191750355394,9.955900737399393,1.9847463725666075,80.23048958876555,1.8583843802388507,5.0555639692810095\nguacamole,241.48575754462223,10.391336110722236,27.55566811101696,9.779999599624496,2.032783537427875,81.61253439454784,1.7750477285359194,5.606711388831522\ngyoza,243.90806293808384,10.168226135294338,27.980666169409588,10.321189180455386,2.0184168728528387,77.97899545754322,1.806320583299712,5.580133199589227\nhamburger,303.27809483067415,11.653377370461147,27.63524067709011,18.106418253436114,1.9938057055424951,80.24541186989481,1.8583843802388507,4.930659157140015\nhot_and_sour_soup,240.8114982533098,10.033841157010114,28.921126874768152,10.321189180455386,2.0006368050141377,80.1325908271569,1.7950527595535715,5.4208925356076545\nhot_dog,293.19638638923954,11.653377370461147,27.947829379511592,18.21700395187885,1.9965441452510064,79.59227646729306,1.8583843802388507,5.19597915582178\nhuevos_rancheros,240.48168501668874,9.989117681055413,28.10805739713372,9.837941814795839,1.99334494725355,79.38260908831622,1.8243500323973525,4.687551633881487\nhummus,235.90128437536782,10.336958007019653,28.921126874768152,10.321189180455386,1.9848900750324676,80.56433500165811,1.775859194511599,4.408490826808424\nice_cream,357.21733290195664,9.411888840460609,52.00664664506674,15.63292563773649,1.8564759064282579,102.11542574279352,1.799674831602202,3.4859359741008538\nlasagna,355.86809472850035,9.725073182838901,38.520834668961356,10.012178062920732,2.013519823561327,102.11542574279352,1.799674831602202,4.750248699715724\nlobster_bisque,237.43646132046007,11.653377370461147,27.77845437068165,9.97824513297501,1.9938057055424951,81.00470946359111,1.805745656796309,5.240983658638862\nlobster_roll_sandwich,262.2115274135363,11.653377370461147,28.921126874768152,10.048254073301027,2.0449121572319395,79.83934619345058,1.7842279032650816,4.473674123144147\nmacaroni_and_cheese,234.7130116158322,10.186212084578699,29.0807535926554,10.187433438602621,2.0015421273744303,102.11542574279352,1.7924785259188596,4.903376313800051\nmacarons,360.5569818259019,9.989117681055413,51.81357848907283,15.901340310882441,1.8564759064282579,92.13824562222432,1.737752244686075,3.4859359741008538\nmiso_soup,239.2038119618733,10.080828273221089,27.592707333690704,9.675206447985111,2.000559517233808,80.56433500165811,1.8039084530544418,4.852302402117236\nmussels,237.57507660144603,11.653377370461147,28.571350806319835,9.980355553373357,1.9915371340146375,80.47729036931709,1.7979208383155516,5.155256999325147\nnachos,254.31300795418076,10.127611866177164,28.921126874768152,10.074836676869,1.9698110672382947,80.64341550843471,1.7949426469259737,4.997096704919089\nomelette,240.1983188790738,9.869389134823333,27.786505223295393,10.031174495267374,1.990034117818239,79.07849207370552,1.7916432855073179,5.026237519359403\nonion_rings,328.80914536359654,10.202956229030775,30.055233231414025,18.390310233577704,2.0063388571221568,78.45350561759527,1.7922850348174724,5.062738920392597\noysters,240.64787302269963,11.653377370461147,27.34167895594095,9.934275001473146,1.9742328342654523,79.78267042096036,1.8148487032893466,5.234677152083395\npad_thai,234.713643912651,10.02753745106709,27.445149391538312,10.089256610157275,1.994153126302977,80.56433500165811,1.799674831602202,4.56403366128047\npaella,257.70307065669834,9.989117681055413,38.0331754372176,10.321189180455386,1.9938057055424951,79.47053152318097,1.8309090582553438,4.903376313800051\npancakes,353.8267731300917,8.448368289511425,51.8948450561462,15.98443019274931,1.8564759064282579,91.38282668344965,1.737752244686075,3.4859359741008538\npanna_cotta,360.7886201635483,8.448368289511425,51.93531625969886,10.321189180455386,1.8564759064282579,89.33176110493842,1.737752244686075,3.4859359741008538\npeking_duck,300.7607096743958,11.653377370461147,27.35825443167446,10.321189180455386,2.006061300653853,80.55795208306776,1.8583843802388507,4.891832530360663\npho,260.7580887150503,9.80562553819056,38.01488602788525,10.155491763227854,1.9754855742419657,80.56433500165811,1.781654336140582,4.463043406172336\npizza,257.10765025992566,9.784981902751438,38.26862693707478,9.98989620158203,1.966720796009269,102.11542574279352,1.8052153417832488,4.937472440965884\npork_chop,300.1461701440697,11.653377370461147,28.921126874768152,18.030594623225653,2.0217077213372567,80.64479780293696,1.8583843802388507,4.762223996953625\npoutine,258.45280188728594,9.802713145794277,27.44724662027197,10.026907957005886,2.040053153405015,79.88158078672174,1.7782237754919872,5.390313551813552\nprime_rib,240.60709896984775,10.269769329981992,27.632429560065006,10.238438256479158,1.968286731669564,80.41738928002768,1.7764692497473535,4.94412476238962\npulled_pork_sandwich,300.6422793371928,11.653377370461147,38.89462655540971,17.77980894349634,1.9938057055424951,79.2225406392878,1.799674831602202,4.903376313800051\nramen,258.45280188728594,9.816739005971575,37.98498881940963,10.169210227879693,1.9995062371021017,80.22853665803818,1.8010055233005917,5.127948747200436\nravioli,258.42572898568994,9.909133026862527,54.41720237832857,9.835867602696721,2.0256595928119903,77.01952701495507,1.8013480400650959,5.362010057575842\nred_velvet_cake,258.45280188728594,8.448368289511425,52.25460665767111,15.969558178485933,1.8564759064282579,88.7336616044647,1.737752244686075,3.4859359741008538\nrisotto,257.2467491634057,10.087784082749046,38.077059785289435,9.90161825995889,2.0270688180098144,80.56433500165811,1.8092174380012112,4.812775382512448\nsamosa,327.1821345998118,9.895975630977434,29.822497970296183,17.913880103479535,2.016955875422883,81.21366594270263,1.8255437005070263,4.908562527126842\nsashimi,246.11601113622893,11.653377370461147,27.976652965979486,9.607237531861827,1.9920861815218356,79.57583376734735,1.790516673323921,5.34695900065755\nscallops,244.9612118328294,11.653377370461147,28.447606623488724,9.935521469031464,2.0129832670463137,80.32585095137237,1.7891605417763978,4.306611372695636\nseaweed_salad,179.58560079303638,9.989117681055413,24.057888495768083,9.972549124990001,1.9938057055424951,77.62072579072066,1.799674831602202,6.091555792958139\nshrimp_and_grits,237.45916712385522,9.989117681055413,27.30800830454291,9.942939814915214,2.039877341041695,79.26085035482126,1.8067696205514647,5.069680315622956\nspaghetti_bolognese,260.6081145253035,10.009031421137998,38.41496433937312,9.858332459912429,2.0213651324182846,81.1499002860831,1.817925807597082,5.0243997254121044\nspaghetti_carbonara,258.2643656751826,9.778948383244051,37.518998836764474,9.995014369196483,1.9938057055424951,81.40373567918625,1.834827474989536,4.903376313800051\nspring_rolls,183.7220009651408,9.857961371816078,24.500981001399087,9.842242982734076,2.115581452707587,79.26785515716388,1.808622961020134,6.091555792958139\nsteak,298.495077140545,11.653377370461147,28.163123369162086,18.104590865623457,1.9536404862830064,78.98046424697661,1.8583843802388507,4.767768058159727\nstrawberry_shortcake,361.641021686838,8.448368289511425,51.625326537642955,15.90590335745124,1.8564759064282579,90.02278504752417,1.737752244686075,3.4859359741008538\nsushi,240.0677789298663,11.653377370461147,27.627395671049197,9.840937167397282,1.9746457557513393,78.9103015970791,1.8090740085466723,5.381141187467261\ntacos,258.95288600582916,10.127583634533858,37.61746377059224,10.321189180455386,2.0659480544125532,81.35060762661348,1.8002478140838187,4.530174361903669\ntakoyaki,330.559826483084,9.981391349292242,30.110786004690574,17.967623016501065,1.9924946176851253,80.56433500165811,1.7889706011446285,5.469104598432647\ntiramisu,358.17666499376213,8.448368289511425,51.843497465246166,16.152468768965495,1.8564759064282579,91.74152786799696,1.737752244686075,3.4859359741008538\ntuna_tartare,238.39810024776227,11.653377370461147,28.02791784404456,10.264388874534328,1.9734657337448958,79.08146175523984,1.8290401366102151,4.763179591851535\nwaffles,258.45280188728594,8.448368289511425,52.02591595311778,16.087128679077765,1.8564759064282579,91.15079317953175,1.737752244686075,3.4859359741008538\n'

Path(AKG_NORMAL_PATH).write_text(AKG_NORMAL_CSV, encoding="utf-8")
Path(AKG_PREGNANT_PATH).write_text(AKG_PREGNANT_CSV, encoding="utf-8")
Path(AKG_BREASTFEEDING_PATH).write_text(AKG_BREASTFEEDING_CSV, encoding="utf-8")
Path(NUTRITION_TABLE_PATH).write_text(NUTRITION_TABLE_CLEANED_CSV, encoding="utf-8")

print("File AKG dan nutrition_table_cleaned.csv berhasil dibuat di:", PROJECT_DIR)

nutrition_df_raw = pd.read_csv(NUTRITION_TABLE_PATH)
akg_normal_df = pd.read_csv(AKG_NORMAL_PATH)
akg_pregnant_df = pd.read_csv(AKG_PREGNANT_PATH)
akg_breastfeeding_df = pd.read_csv(AKG_BREASTFEEDING_PATH)

print("Nutrition table shape:", nutrition_df_raw.shape)
print("AKG normal shape:", akg_normal_df.shape)

display(nutrition_df_raw.head())
display(akg_normal_df.head())


File AKG dan nutrition_table_cleaned.csv berhasil dibuat di: /content/nutrivision_cnn_food101_akg
Nutrition table shape: (101, 9)
AKG normal shape: (26, 45)


,class_name,calories_kcal,protein_g,carbs_g,fat_g,fiber_g,calcium_mg,iron_mg,vitamin_c_mg
0,apple_pie,360.531380,8.448368,51.013617,16.042086,1.856476,80.564335,1.737752,3.485936
1,baby_back_ribs,300.050396,11.653377,28.921127,18.032969,1.981891,79.009366,1.858384,4.800256
2,baklava,359.228632,8.448368,28.921127,15.922833,1.856476,89.826141,1.799675,3.485936
3,beef_carpaccio,301.734171,11.653377,27.987210,17.759368,1.982885,78.561907,1.858384,4.939265
4,beef_tartare,304.032317,11.653377,28.921127,18.151466,1.994065,80.685123,1.858384,5.284609


,age_category,age_group,body_weight,height,calories,protein,total_fat,omega_3,omega_6,carbohydrates,...,sodium,chlorine,copper,vitamin_a,min_age,max_age,preg_month_min,preg_month_max,bf_month_min,bf_month_max
0,infants_children,0-5 months,6.0,60.0,550.0,9.0,31.0,0.5,4.4,59.0,...,120.0,180.0,0.20,375.0,0.0,0.416667,0.0,0.0,0.0,0.0
1,infants_children,6-11 months,9.0,72.0,800.0,15.0,35.0,0.5,4.4,105.0,...,370.0,570.0,0.22,400.0,0.5,0.916667,0.0,0.0,0.0,0.0
2,infants_children,1-3 years,13.0,92.0,1350.0,20.0,45.0,0.7,7.0,215.0,...,800.0,1200.0,0.34,400.0,1.0,3.000000,0.0,0.0,0.0,0.0
3,infants_children,4-6 years,19.0,113.0,1400.0,25.0,50.0,0.9,10.0,220.0,...,900.0,1300.0,0.44,450.0,4.0,6.000000,0.0,0.0,0.0,0.0
4,infants_children,7-9 years,27.0,130.0,1650.0,40.0,55.0,0.9,10.0,250.0,...,1000.0,1500.0,0.57,500.0,7.0,9.000000,0.0,0.0,0.0,0.0


In [ ]:
# ======================
# Load dataset Food-101
# ======================

try:
    read_config = tfds.ReadConfig(shuffle_seed=SEED, try_autocaching=False)
except TypeError:
    read_config = tfds.ReadConfig(shuffle_seed=SEED)

(ds_train_raw, ds_val_raw), info = tfds.load(
    "food101",
    split=["train", "validation"],
    as_supervised=True,
    with_info=True,
    shuffle_files=True,
    read_config=read_config,
    data_dir="/content/tfds"
)

all_class_names = info.features["label"].names
assert len(all_class_names) == 101

class_names = all_class_names[:CLASS_LIMIT]
NUM_CLASSES = len(class_names)

with open(CLASS_NAMES_PATH, "w") as f:
    json.dump(class_names, f, indent=2)

print("Jumlah kelas:", NUM_CLASSES)
print("Contoh kelas:", class_names[:10])
show_ram("Setelah load TFDS")


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /content/tfds/food101/incomplete.HBRPOI_2.0.0/food101-train.tfrecord-[0-9][0-9][0-9][0-9][0-9]-of-[0…

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /content/tfds/food101/incomplete.HBRPOI_2.0.0/food101-validation.tfrecord-[0-9][0-9][0-9][0-9][0-9]-…

Dataset food101 downloaded and prepared to /content/tfds/food101/2.0.0. Subsequent calls will reuse this data.
Jumlah kelas: 101
Contoh kelas: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito']
[Setelah load TFDS] used=2.44/167.05 GB, available=163.08 GB, percent=2.4%


In [ ]:
# ============================================================
# Menyiapkan nutrition_table_cleaned sesuai urutan kelas Food-101
# ============================================================

NUTRIENT_NAMES = [
    "calories_kcal",
    "protein_g",
    "carbs_g",
    "fat_g",
    "fiber_g",
    "calcium_mg",
    "iron_mg",
    "vitamin_c_mg",
]

# Batas normalisasi output nutrisi per 100 gram. Dipakai untuk stabilitas training, bukan sebagai AKG.
NUTRIENT_MAX = np.array(
    [1000, 100, 160, 100, 50, 1600, 30, 200],
    dtype=np.float32
)

nutrition_df = pd.read_csv(NUTRITION_TABLE_PATH)

required_cols = ["class_name"] + NUTRIENT_NAMES
missing_cols = [c for c in required_cols if c not in nutrition_df.columns]
if missing_cols:
    raise ValueError(f"Kolom nutrition_table_cleaned.csv kurang: {missing_cols}")

missing_foods = sorted(set(class_names) - set(nutrition_df["class_name"].tolist()))
if missing_foods:
    raise ValueError(f"Kelas Food-101 berikut tidak ada di nutrition table: {missing_foods[:10]}")

nutrition_df = (
    nutrition_df
    .set_index("class_name")
    .loc[class_names]
    .reset_index()
)

nutrition_table = nutrition_df[NUTRIENT_NAMES].astype(np.float32).values
nutrition_table_norm = np.clip(nutrition_table / NUTRIENT_MAX, 0.0, 1.0).astype(np.float32)

nutrition_df.to_csv(NUTRITION_TABLE_PATH, index=False)

print("Nutrition table siap:", nutrition_table.shape)
display(nutrition_df.head())


Nutrition table siap: (101, 8)


,class_name,calories_kcal,protein_g,carbs_g,fat_g,fiber_g,calcium_mg,iron_mg,vitamin_c_mg
0,apple_pie,360.531380,8.448368,51.013617,16.042086,1.856476,80.564335,1.737752,3.485936
1,baby_back_ribs,300.050396,11.653377,28.921127,18.032969,1.981891,79.009366,1.858384,4.800256
2,baklava,359.228632,8.448368,28.921127,15.922833,1.856476,89.826141,1.799675,3.485936
3,beef_carpaccio,301.734171,11.653377,27.987210,17.759368,1.982885,78.561907,1.858384,4.939265
4,beef_tartare,304.032317,11.653377,28.921127,18.151466,1.994065,80.685123,1.858384,5.284609


In [ ]:
# ==========================================
# Fungsi AKG sebagai acuan kebutuhan harian
# ==========================================

AKG_TO_NUTRIENT_MAP = {
    "calories_kcal": "calories",
    "protein_g": "protein",
    "carbs_g": "carbohydrates",
    "fat_g": "total_fat",
    "fiber_g": "dietary_fiber",
    "calcium_mg": "calcium",
    "iron_mg": "iron",
    "vitamin_c_mg": "vitamin_c",
}

def get_base_akg_row(age, sex="male"):
    df = pd.read_csv(AKG_NORMAL_PATH)
    age = float(age)
    sex = str(sex).lower().strip()

    if age < 10:
        subset = df[
            (df["age_category"] == "infants_children") &
            (df["min_age"] <= age) &
            (df["max_age"] >= age)
        ]
    else:
        subset = df[
            (df["age_category"] == sex) &
            (df["min_age"] <= age) &
            (df["max_age"] >= age)
        ]

    if len(subset) == 0:
        raise ValueError(f"AKG dasar tidak ditemukan untuk age={age}, sex={sex}")

    return subset.iloc[0].copy()

def get_increment_akg_row(status="normal", pregnancy_month=0, breastfeeding_month=0):
    status = str(status).lower().strip()

    if status == "pregnant":
        df = pd.read_csv(AKG_PREGNANT_PATH)
        m = float(pregnancy_month)
        subset = df[
            (df["age_category"] == "pregnant") &
            (df["preg_month_min"] <= m) &
            (df["preg_month_max"] >= m)
        ]
    elif status == "breastfeeding":
        df = pd.read_csv(AKG_BREASTFEEDING_PATH)
        m = float(breastfeeding_month)
        subset = df[
            (df["age_category"] == "breastfeeding") &
            (df["bf_month_min"] <= m) &
            (df["bf_month_max"] >= m)
        ]
    else:
        return None

    if len(subset) == 0:
        raise ValueError(
            f"Tambahan AKG tidak ditemukan untuk status={status}, "
            f"pregnancy_month={pregnancy_month}, breastfeeding_month={breastfeeding_month}"
        )

    return subset.iloc[0].copy()

def get_daily_requirement_from_akg(
    age=21,
    sex="male",
    status="normal",
    pregnancy_month=0,
    breastfeeding_month=0
):
    status = str(status).lower().strip()
    sex = str(sex).lower().strip()

    if status in ["pregnant", "breastfeeding"]:
        sex = "female"

    base = get_base_akg_row(age=age, sex=sex)
    inc = get_increment_akg_row(
        status=status,
        pregnancy_month=pregnancy_month,
        breastfeeding_month=breastfeeding_month
    )

    requirement = {}
    for nutrient_name, akg_col in AKG_TO_NUTRIENT_MAP.items():
        base_value = float(base[akg_col])
        inc_value = 0.0 if inc is None else float(inc[akg_col])
        requirement[nutrient_name] = base_value + inc_value

    return requirement

def classify_deficiency_risk_by_akg(
    total_nutrition,
    age=21,
    sex="male",
    status="normal",
    pregnancy_month=0,
    breastfeeding_month=0
):
    daily_req = get_daily_requirement_from_akg(
        age=age,
        sex=sex,
        status=status,
        pregnancy_month=pregnancy_month,
        breastfeeding_month=breastfeeding_month
    )

    rows = []
    for nutrient, requirement in daily_req.items():
        intake = float(total_nutrition.get(nutrient, 0.0))
        ratio = intake / requirement if requirement > 0 else 0.0

        if ratio < 0.50:
            risk = "Tinggi"
        elif ratio < 0.80:
            risk = "Sedang"
        else:
            risk = "Rendah"

        rows.append({
            "nutrient": nutrient,
            "intake": intake,
            "daily_requirement_akg": requirement,
            "fulfillment_ratio": ratio,
            "deficiency_risk": risk
        })

    return pd.DataFrame(rows)

display(pd.DataFrame([get_daily_requirement_from_akg(age=21, sex="male", status="normal")]))


,calories_kcal,protein_g,carbs_g,fat_g,fiber_g,calcium_mg,iron_mg,vitamin_c_mg
0,2650.0,65.0,430.0,75.0,37.0,1000.0,9.0,90.0


In [ ]:
# ============================================================
# Data pipeline Food-101 + target nutrisi dari nutrition table
# ============================================================

nutrition_table_norm_tf = tf.constant(nutrition_table_norm, dtype=tf.float32)

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)

    y_class = tf.one_hot(label, depth=NUM_CLASSES, dtype=tf.float32)
    y_nutrition = tf.gather(nutrition_table_norm_tf, label)

    y = {
        "class_output": y_class,
        "nutrition_output": y_nutrition
    }

    return image, y

def prepare_dataset(ds, training=True):
    if training:
        ds = ds.shuffle(SHUFFLE_BUFFER, seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(preprocess, num_parallel_calls=PARALLEL_CALLS)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(PREFETCH_BUFFER)

    return ds

train_ds = prepare_dataset(ds_train_raw, training=True)
val_ds = prepare_dataset(ds_val_raw, training=False)

for images, labels in train_ds.take(1):
    print("Image batch:", images.shape)
    print("Class target:", labels["class_output"].shape)
    print("Nutrition target:", labels["nutrition_output"].shape)

show_ram("Setelah dataset pipeline")


Image batch: (32, 224, 224, 3)
Class target: (32, 101)
Nutrition target: (32, 8)
[Setelah dataset pipeline] used=3.65/167.05 GB, available=161.75 GB, percent=3.2%


In [ ]:
import tensorflow as tf

# ==============================
# Custom Layer dan Custom Loss
# ==============================

@tf.keras.utils.register_keras_serializable(package="NutriVision")
class NutritionFromClassProbability(tf.keras.layers.Layer):
    """
    Custom Layer:
    Mengubah probabilitas kelas makanan menjadi estimasi nutrisi berdasarkan
    nutrition_table_cleaned.csv yang sudah dinormalisasi.
    """
    def __init__(self, nutrition_table_norm, **kwargs):
        super().__init__(**kwargs)
        self.nutrition_table_norm_list = np.asarray(
            nutrition_table_norm,
            dtype=np.float32
        ).tolist()

    def build(self, input_shape):
        nutrition_array = np.asarray(self.nutrition_table_norm_list, dtype=np.float32)
        self.nutrition_lookup = self.add_weight(
            name="nutrition_lookup",
            shape=nutrition_array.shape,
            initializer=tf.constant_initializer(nutrition_array),
            trainable=False,
            dtype=tf.float32
        )

    def call(self, class_probs):
        class_probs = tf.cast(class_probs, tf.float32)
        return tf.matmul(class_probs, self.nutrition_lookup)

    def get_config(self):
        config = super().get_config()
        config.update(
            {"nutrition_table_norm": self.nutrition_table_norm_list}
        )
        return config

@tf.keras.utils.register_keras_serializable(package="NutriVision")
class WeightedNutritionMAELoss(tf.keras.losses.Loss):
    """
    Custom Loss:
    MAE berbobot untuk prediksi nutrisi.
    """
    def __init__(self, nutrient_weights=None, name="weighted_nutrition_mae_loss", **kwargs):
        super().__init__(name=name, **kwargs)

        if nutrient_weights is None:
            nutrient_weights = [1.0] * len(NUTRIENT_NAMES)

        self.nutrient_weights = list(nutrient_weights)

    def call(self, y_true, y_pred):
        weights = tf.constant(self.nutrient_weights, dtype=tf.float32)
        abs_error = tf.abs(tf.cast(y_true, tf.float32) - tf.cast(y_pred, tf.float32))
        weighted_error = abs_error * weights
        return tf.reduce_mean(
            tf.reduce_sum(weighted_error, axis=-1) / tf.reduce_sum(weights)
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {"nutrient_weights": self.nutrient_weights}
        )
        return config

In [ ]:
# =================================================
# Custom CNN model dengan TensorFlow Functional API
# =================================================

tf.keras.backend.clear_session()

def conv_bn_act(x, filters, kernel_size=3, strides=1, name=None):
    x = tf.keras.layers.Conv2D(
        filters,
        kernel_size,
        strides=strides,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=None if name is None else name + "_conv"
    )(x)
    x = tf.keras.layers.BatchNormalization(
        name=None if name is None else name + "_bn"
    )(x)
    x = tf.keras.layers.Activation(
        "swish",
        name=None if name is None else name + "_swish"
    )(x)
    return x

def squeeze_excite(x, ratio=8, name="se"):
    filters = int(x.shape[-1])
    se = tf.keras.layers.GlobalAveragePooling2D(name=name + "_gap")(x)
    se = tf.keras.layers.Dense(
        max(filters // ratio, 8),
        activation="swish",
        name=name + "_fc1"
    )(se)
    se = tf.keras.layers.Dense(
        filters,
        activation="sigmoid",
        name=name + "_fc2"
    )(se)
    se = tf.keras.layers.Reshape((1, 1, filters), name=name + "_reshape")(se)
    return tf.keras.layers.Multiply(name=name + "_scale")([x, se])

def ds_block(x, filters, strides=1, dropout=0.0, name="ds"):
    shortcut = x

    x = tf.keras.layers.DepthwiseConv2D(
        3,
        strides=strides,
        padding="same",
        use_bias=False,
        depthwise_initializer="he_normal",
        name=name + "_dw"
    )(x)
    x = tf.keras.layers.BatchNormalization(name=name + "_dw_bn")(x)
    x = tf.keras.layers.Activation("swish", name=name + "_dw_swish")(x)

    x = tf.keras.layers.Conv2D(
        filters,
        1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=name + "_pw"
    )(x)
    x = tf.keras.layers.BatchNormalization(name=name + "_pw_bn")(x)
    x = tf.keras.layers.Activation("swish", name=name + "_pw_swish")(x)

    x = squeeze_excite(x, ratio=8, name=name + "_se")

    if dropout > 0:
        x = tf.keras.layers.Dropout(dropout, name=name + "_drop")(x)

    if strides == 1 and int(shortcut.shape[-1]) == filters:
        x = tf.keras.layers.Add(name=name + "_add")([shortcut, x])

    return x

def build_nutrivision_cnn(input_shape=(224, 224, 3), num_classes=101):
    inputs = tf.keras.Input(shape=input_shape, name="image_input")

    x = tf.keras.layers.RandomFlip("horizontal", name="aug_flip")(inputs)
    x = tf.keras.layers.RandomRotation(0.08, name="aug_rotate")(x)
    x = tf.keras.layers.RandomZoom(0.12, name="aug_zoom")(x)
    x = tf.keras.layers.RandomContrast(0.15, name="aug_contrast")(x)

    x = tf.keras.layers.Rescaling(1.0 / 255.0, name="rescale")(x)

    x = conv_bn_act(x, 32, 3, 2, name="stem")

    x = ds_block(x, 48, strides=1, dropout=0.02, name="block1_1")
    x = ds_block(x, 64, strides=2, dropout=0.03, name="block2_1")
    x = ds_block(x, 64, strides=1, dropout=0.03, name="block2_2")

    x = ds_block(x, 96, strides=2, dropout=0.04, name="block3_1")
    x = ds_block(x, 96, strides=1, dropout=0.04, name="block3_2")

    x = ds_block(x, 144, strides=2, dropout=0.05, name="block4_1")
    x = ds_block(x, 144, strides=1, dropout=0.05, name="block4_2")

    x = ds_block(x, 192, strides=2, dropout=0.06, name="block5_1")
    x = ds_block(x, 192, strides=1, dropout=0.06, name="block5_2")

    x = ds_block(x, 256, strides=2, dropout=0.06, name="block6_1")
    x = ds_block(x, 256, strides=1, dropout=0.06, name="block6_2")

    x = conv_bn_act(x, 384, 1, 1, name="final_conv")
    x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = tf.keras.layers.Dropout(0.35, name="head_dropout")(x)

    shared = tf.keras.layers.Dense(512, activation="swish", name="shared_dense")(x)
    shared = tf.keras.layers.Dropout(0.30, name="shared_dropout")(shared)

    class_output = tf.keras.layers.Dense(
        num_classes,
        activation="softmax",
        dtype="float32",
        name="class_output"
    )(shared)

    nutrition_output = NutritionFromClassProbability(
        nutrition_table_norm=nutrition_table_norm,
        name="nutrition_output"
    )(class_output)

    return tf.keras.Model(
        inputs=inputs,
        outputs={
            "class_output": class_output,
            "nutrition_output": nutrition_output
        },
        name="NutriVision_CNN_Food101_AKG"
    )

model = build_nutrivision_cnn(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    num_classes=NUM_CLASSES
)

model.summary()


Model: "NutriVision_CNN_Food101_AKG"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aug_flip            │ (None, 224, 224,  │          0 │ image_input[0][0] │
│ (RandomFlip)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aug_rotate          │ (None, 224, 224,  │          0 │ aug_flip[0][0]    │
│ (RandomRotation)    │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aug_zoom            │ (None, 224, 224,  │          0 │ aug_rotate[0][0]  │
│ (RandomZoom)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aug_contrast        │ (None, 224, 224,  │          0 │ aug_zoom[0][0]    │
│ (RandomContrast)    │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescale (Rescaling) │ (None, 224, 224,  │          0 │ aug_contrast[0][… │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ rescale[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_swish          │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_1_dw         │ (None, 112, 112,  │        288 │ stem_swish[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_1_dw_bn      │ (None, 112, 112,  │        128 │ block1_1_dw[0][0] │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_1_dw_swish   │ (None, 112, 112,  │          0 │ block1_1_dw_bn[0… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_1_pw         │ (None, 112, 112,  │      1,536 │ block1_1_dw_swis… │
│ (Conv2D)            │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_1_pw_bn      │ (None, 112, 112,  │        192 │ block1_1_pw[0][0] │
│ (BatchNormalizatio… │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_1_pw_swish   │ (None, 112, 112,  │          0 │ block1_1_pw_bn[0… │
│ (Activation)        │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_1_se_gap     │ (None, 48)        │          0 │ block1_1_pw_swis… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_1_se_fc1     │ (None, 8)         │        392 │ block1_1_se_gap[

 Total params: 682,609 (2.60 MB)

 Trainable params: 675,209 (2.58 MB)

 Non-trainable params: 7,400 (28.91 KB)

In [ ]:
# =================
# Custom Callback
# =================

class RAMSafeTargetCallback(tf.keras.callbacks.Callback):
    def __init__(
        self,
        target_accuracy=0.85,
        target_mae=0.02,
        save_path="best_model.keras",
        patience=18,
        lr_patience=6,
        lr_factor=0.5,
        min_lr=1e-6,
        min_available_ram_gb=2.0,
        restore_best_weights=True,
        verbose=1
    ):
        super().__init__()
        self.target_accuracy = target_accuracy
        self.target_mae = target_mae
        self.save_path = save_path
        self.patience = patience
        self.lr_patience = lr_patience
        self.lr_factor = lr_factor
        self.min_lr = min_lr
        self.min_available_ram_gb = min_available_ram_gb
        self.restore_best_weights = restore_best_weights
        self.verbose = verbose

        self.best_score = -np.inf
        self.best_weights = None
        self.wait = 0
        self.lr_wait = 0

    def _get_log_value(self, logs, candidates):
        logs = logs or {}
        for key in candidates:
            if key in logs:
                return float(logs[key])
        return None

    def _get_lr(self):
        try:
            return float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        except Exception:
            return None

    def _set_lr(self, new_lr):
        try:
            tf.keras.backend.set_value(self.model.optimizer.learning_rate, new_lr)
        except Exception:
            self.model.optimizer.learning_rate = new_lr

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        mem = psutil.virtual_memory()
        available_gb = mem.available / 1024**3

        if self.verbose:
            print(
                f"[RAMSafeCallback] RAM used={mem.used/1024**3:.2f}/{mem.total/1024**3:.2f} GB, "
                f"available={available_gb:.2f} GB, percent={mem.percent:.1f}%"
            )

        if available_gb < self.min_available_ram_gb:
            print("[RAMSafeCallback] RAM hampir habis. Training dihentikan aman.")
            self.model.stop_training = True
            return

        val_acc = self._get_log_value(logs, [
            "val_class_output_accuracy",
            "val_class_output_categorical_accuracy",
            "val_accuracy"
        ])

        val_mae = self._get_log_value(logs, [
            "val_nutrition_output_mae",
            "val_nutrition_output_mean_absolute_error"
        ])

        if val_acc is None or val_mae is None:
            return

        score = val_acc - val_mae

        if score > self.best_score:
            self.best_score = score
            self.best_weights = self.model.get_weights()
            self.wait = 0
            self.lr_wait = 0
            self.model.save(self.save_path)

            if self.verbose:
                print(
                    f"[RAMSafeCallback] Model terbaik disimpan. "
                    f"val_acc={val_acc:.4f}, val_mae={val_mae:.4f}, score={score:.4f}"
                )
        else:
            self.wait += 1
            self.lr_wait += 1

        if val_acc >= self.target_accuracy and val_mae <= self.target_mae:
            print(
                f"[RAMSafeCallback] Target tercapai: "
                f"val_acc={val_acc:.4f}, val_mae={val_mae:.4f}"
            )
            self.model.stop_training = True
            return

        if self.lr_wait >= self.lr_patience:
            old_lr = self._get_lr()
            if old_lr is not None:
                new_lr = max(old_lr * self.lr_factor, self.min_lr)
                self._set_lr(new_lr)
                print(f"[RAMSafeCallback] Learning rate turun: {old_lr:.8f} -> {new_lr:.8f}")
            self.lr_wait = 0

        if self.wait >= self.patience:
            print("[RAMSafeCallback] Early stopping aktif.")
            if self.restore_best_weights and self.best_weights is not None:
                self.model.set_weights(self.best_weights)
                print("[RAMSafeCallback] Bobot terbaik dikembalikan.")
            self.model.stop_training = True


In [ ]:
# ============================
# Compile dan training model
# ============================

try:
    optimizer = tf.keras.optimizers.AdamW(
        learning_rate=BASE_LR,
        weight_decay=WEIGHT_DECAY
    )
except Exception:
    optimizer = tf.keras.optimizers.Adam(learning_rate=BASE_LR)

model.compile(
    optimizer=optimizer,
    loss={
        "class_output": tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        "nutrition_output": WeightedNutritionMAELoss()
    },
    loss_weights={
        "class_output": 1.0,
        "nutrition_output": 0.10
    },
    metrics={
        "class_output": [
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
        "nutrition_output": [
            tf.keras.metrics.MeanAbsoluteError(name="mae")
        ]
    }
)

callbacks = [
    RAMSafeTargetCallback(
        target_accuracy=TARGET_ACCURACY,
        target_mae=TARGET_MAE,
        save_path=BEST_MODEL_PATH,
        patience=18,
        lr_patience=6,
        lr_factor=0.5,
        min_lr=1e-6,
        min_available_ram_gb=2.0,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        str(PROJECT_DIR / "training_log.csv"),
        append=False
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)


Epoch 1/100
2368/2368 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - class_output_accuracy: 0.0300 - class_output_loss: 4.4947 - loss: 4.4965 - nutrition_output_loss: 0.0182 - nutrition_output_mae: 0.0182[RAMSafeCallback] RAM used=5.01/167.05 GB, available=160.26 GB, percent=4.1%
[RAMSafeCallback] Model terbaik disimpan. val_acc=0.1177, val_mae=0.0167, score=0.1010
2368/2368 ━━━━━━━━━━━━━━━━━━━━ 198s 73ms/step - class_output_accuracy: 0.0542 - class_output_loss: 4.3187 - loss: 4.3207 - nutrition_output_loss: 0.0178 - nutrition_output_mae: 0.0178 - val_class_output_accuracy: 0.1177 - val_class_output_loss: 3.8882 - val_loss: 3.8908 - val_nutrition_output_loss: 0.0167 - val_nutrition_output_mae: 0.0167
Epoch 2/100
2367/2368 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - class_output_accuracy: 0.1162 - class_output_loss: 3.9145 - loss: 3.9162 - nutrition_output_loss: 0.0168 - nutrition_output_mae: 0.0168[RAMSafeCallback] RAM used=5.23/167.05 GB, available=160.04 GB, percent=4.2%
[RAMSafeCallback] Model terbaik

In [ ]:
# ===========================
# Download artifact deploy
# ===========================

# Isi zip:
# - final/best .keras
# - SavedModel
# - class_names.json
# - nutrition_table_cleaned.csv
# - AKG csv
# - training_log.csv

!cd /content && zip -qr nutrivision_cnn_food101_akg_cleaned_artifacts.zip nutrivision_cnn_food101_akg

from google.colab import files
files.download("/content/nutrivision_cnn_food101_akg_cleaned_artifacts.zip")
